# Data export

All results of a recording live in its OME-Zarr store (`<name>.ome.zarr`, next to the TIFF), which opens in any
Zarr reader. For spreadsheets and statistics, SarcAsM exports the features as `.xlsx`, `.csv` or `.json`; the
format follows the file extension. The GUI's export buttons run the same functions.

This tutorial uses the recordings analysed in the [structure](tutorial_structure_analysis.ipynb) and
[tracking](tutorial_sarcomere_tracking.ipynb) tutorials.

In [1]:
import os
import re

import pandas as pd
from sarcasm import SarcAsM, Export, BatchExport

out = 'export_example/'
os.makedirs(out, exist_ok=True)

## Structure features of one recording

`Export.export_structure_data` writes one value per feature and frame. `structure_keys` selects the features (all
keys: [structure features](../structure_features.rst)); the default is `Export.structure_keys_default`. An
`.xlsx` file also gets a `metadata` sheet and a `metrics` sheet with each feature's name, unit and description.
`raw=True` writes the per-object distributions (every Z-band, every vector) and needs `.json`.

In [2]:
sarc = SarcAsM('../../test_data/long_term_2D_ACTN2-citrine_CM/20211115_ACTN2_CMs_96well_control_12days.tif')

Export.export_structure_data(sarc, out + 'structure.xlsx')
Export.export_structure_data(sarc, out + 'structure_selected.csv',
                             structure_keys=['structure.sarcomere.slen_mean', 'structure.sarcomere.oop'])
Export.export_structure_data(sarc, out + 'structure_raw.json', raw=True)

pd.read_excel(out + 'structure.xlsx', sheet_name=None).keys()

2026-10-31 15:31:50 - sarcasm.utils - INFO - Using device: mps


dict_keys(['data', 'metadata', 'metrics'])

## Motion features of one recording

`Export.export_motion_data` writes one row per group of one grouping kind; per-cycle features are averaged per
group. `motion_keys` selects features by their name within the kind (`'beating_rate'`, `'contr_max'`, ...; all:
[motion features](../motion_features.rst)).

In [3]:
sarc = SarcAsM('../../test_data/high_speed_single_ACTN2-citrine_CM/20kPa.tif')

n = Export.export_motion_data(sarc, out + 'motion_myofibril.xlsx', kind='myofibril')
print(f'{n} myofibrils written')
Export.export_motion_data(sarc, out + 'motion_pool.json', kind='pool', motion_keys=['beating_rate', 'contr_max'])

pd.read_excel(out + 'motion_myofibril.xlsx')[['group_id', 'group_member_count', 'beating_rate', 'contr_max',
                                                'time_to_peak', 'popping_rate']].head()

2026-10-31 15:31:53 - sarcasm.utils - INFO - Using device: mps


64 myofibrils written


,group_id,group_member_count,beating_rate,contr_max,time_to_peak,popping_rate
0,0,31,0.472505,-0.097458,0.437494,0.086022
1,1,26,0.472505,-0.102774,0.448432,0.076923
2,2,27,0.478064,-0.080522,0.470306,0.098765
3,3,24,0.478064,-0.108791,0.399214,0.069444
4,4,12,0.478064,-0.136388,0.388276,0.055556


The per-track data are a DataFrame and arrays in the store, so pandas and NumPy export them directly:

In [4]:
sarc.motion.tracks().to_csv(out + 'tracks.csv', index=False)

## Many recordings

`BatchExport` collects the features of many analysed recordings into one table. Keyword arguments add a column
each: a constant, or a function of the file name, e.g. a regular expression for the condition.

In [5]:
files = ['../../test_data/high_speed_single_ACTN2-citrine_CM/20kPa.tif']
stiffness = lambda file: re.search(r'(\d+)kPa', file).group(1)

batch = BatchExport(files, folder=out, experiment='gel stiffness', stiffness=stiffness)
batch.get_motion_data(kinds=['pool', 'myofibril'])     # one row per group and file
batch.export_data(out + 'motion_all.xlsx')
batch.data[['file_name', 'experiment', 'stiffness', 'kind', 'group_id', 'beating_rate', 'contr_max']].head()

  0%|          | 0/1 [00:00<?, ?it/s]

2026-10-31 15:31:53 - sarcasm.utils - INFO - Using device: mps


100%|██████████| 1/1 [00:00<00:00, 16.89it/s]

,file_name,experiment,stiffness,kind,group_id,beating_rate,contr_max
0,20kPa.tif,gel stiffness,20,pool,0,0.476197,-0.117798
1,20kPa.tif,gel stiffness,20,myofibril,0,0.472505,-0.097458
2,20kPa.tif,gel stiffness,20,myofibril,1,0.472505,-0.102774
3,20kPa.tif,gel stiffness,20,myofibril,2,0.478064,-0.080522
4,20kPa.tif,gel stiffness,20,myofibril,3,0.478064,-0.108791


`batch.get_data()` collects the structure features instead (one row per file; a movie's per-frame features are
averaged, and every frame is also written to a `per_frame` sheet). To reuse a table without re-reading the
stores: `get_motion_data` saves it in `folder` and `batch.load_motion_data()` loads it; for the structure table,
`batch.save_data()` and `batch.load_data()`.